In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
import numpy as np
import pandas as pd
import os
from itertools import combinations
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
from scipy.special import logit, expit
from xgboost import XGBClassifier, XGBRegressor
import lightgbm as lgb
import catboost as cb
from sklearn.preprocessing import LabelEncoder, KBinsDiscretizer
from sklearn.linear_model import Ridge
import gc

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)

# Load data
INPUT_DIR = '/kaggle/input/playground-series-s5e11'
train = pd.read_csv(f'{INPUT_DIR}/train.csv')
test = pd.read_csv(f'{INPUT_DIR}/test.csv')
original = pd.read_csv('/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv')
submission = pd.read_csv(f'{INPUT_DIR}/sample_submission.csv')

# Define target
TARGET = 'loan_paid_back'

print(f"Train shape: {train.shape}, Test shape: {test.shape}, Original shape: {original.shape}")

Train shape: (593994, 13), Test shape: (254569, 12), Original shape: (20000, 22)


In [2]:
# Prepare the combined dataset as in winning solution
FEATURES = [col for col in train.columns if col not in ['id', TARGET]]
CATEGORICAL = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
NUMERICAL = [col for col in FEATURES if col not in CATEGORICAL]

print(f"Features: {FEATURES}")
print(f"Categorical: {CATEGORICAL}")
print(f"Numerical: {NUMERICAL}")

# Add source indicator
train['source'] = 'train'
test['source'] = 'test'
original['source'] = 'original'

# Combine all data
combined = pd.concat([train.drop(columns=['id']), 
                      test.drop(columns=['id']), 
                      original], axis=0).reset_index(drop=True)

# Save original target for later
if TARGET in combined.columns:
    y_original = combined[combined['source'] == 'original'][TARGET].copy()

Features: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
Categorical: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
Numerical: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
import numpy as np
import pandas as pd
import os
from itertools import combinations
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
from scipy.special import logit, expit
from xgboost import XGBClassifier
import gc

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)

# Load data
INPUT_DIR = '/kaggle/input/playground-series-s5e11'
train = pd.read_csv(f'{INPUT_DIR}/train.csv')
test = pd.read_csv(f'{INPUT_DIR}/test.csv')
original = pd.read_csv('/kaggle/input/loan-prediction-dataset-2025/loan_dataset_20000.csv')

# Define target
TARGET = 'loan_paid_back'
FEATURES = [col for col in train.columns if col not in ['id', TARGET]]
CATEGORICAL = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
NUMERICAL = [col for col in FEATURES if col not in CATEGORICAL]

print(f"Train shape: {train.shape}, Test shape: {test.shape}, Original shape: {original.shape}")
print(f"Memory usage - Train: {train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Memory usage - Test: {test.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Optimize data types for base data
def optimize_dtypes(df):
    """Downcast numerical columns to save memory"""
    df = df.copy()
    
    # Downcast float64 to float32
    float_cols = df.select_dtypes(include=['float64']).columns
    for col in float_cols:
        df[col] = df[col].astype('float32')
    
    # Downcast int64 to int32 or int16
    int_cols = df.select_dtypes(include=['int64']).columns
    for col in int_cols:
        col_min = df[col].min()
        col_max = df[col].max()
        
        if col_min >= -32768 and col_max <= 32767:
            df[col] = df[col].astype('int16')
        elif col_min >= -2147483648 and col_max <= 2147483647:
            df[col] = df[col].astype('int32')
    
    return df

# Optimize base data
train = optimize_dtypes(train)
test = optimize_dtypes(test)
original = optimize_dtypes(original)

# Add source column
train['source'] = 'train'
test['source'] = 'test'
original['source'] = 'original'

# Combine all data
combined = pd.concat([train.drop(columns=['id']), 
                      test.drop(columns=['id']), 
                      original], axis=0).reset_index(drop=True)

# Memory check
print(f"\nCombined shape: {combined.shape}")
print(f"Memory usage: {combined.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
gc.collect()

Train shape: (593994, 13), Test shape: (254569, 12), Original shape: (20000, 22)
Memory usage - Train: 250.50 MB
Memory usage - Test: 105.41 MB

Combined shape: (868563, 23)
Memory usage: 442.28 MB


0

In [4]:
# ============================================
# TARGET ENCODING PIPELINE (MEMORY OPTIMIZED)
# ============================================

class CrossFoldTargetEncoder:
    """Target Encoder with built-in cross-validation to prevent leakage"""
    
    def __init__(self, cols, n_folds=5, smooth=10):
        self.cols = cols
        self.n_folds = n_folds
        self.smooth = smooth
        self.encodings = {}
        self.global_means = {}
        
    def fit_transform(self, X, y):
        X_encoded = X.copy()
        
        # Calculate global means for each column
        for col in self.cols:
            self.global_means[col] = y.mean()
        
        # Create KFold
        kf = StratifiedKFold(n_splits=self.n_folds, shuffle=True, random_state=SEED)
        
        # Process columns one by one to save memory
        for col_idx, col in enumerate(self.cols):
            print(f"Encoding column {col_idx+1}/{len(self.cols)}: {col}")
            
            # Initialize column
            X_encoded[f'TE_{col}'] = 0.0
            
            for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train = y.iloc[train_idx]
                
                # Calculate target mean for each category in training fold
                target_mean = y_train.groupby(X_train[col]).mean()
                counts = X_train[col].value_counts()
                
                # Apply smoothing
                smoothed = (target_mean * counts + self.global_means[col] * self.smooth) / (counts + self.smooth)
                
                # Apply to validation fold
                X_encoded.loc[X_val.index, f'TE_{col}'] = X_val[col].map(smoothed)
                X_encoded.loc[X_val.index, f'TE_{col}'] = X_encoded.loc[X_val.index, f'TE_{col}'].fillna(self.global_means[col])
            
            # Free memory
            gc.collect()
        
        # For training data, also store the full encoding for transform
        for col in self.cols:
            target_mean = y.groupby(X[col]).mean()
            counts = X[col].value_counts()
            smoothed = (target_mean * counts + self.global_means[col] * self.smooth) / (counts + self.smooth)
            self.encodings[col] = smoothed
        
        # Convert to float32 to save memory
        for col in self.cols:
            X_encoded[f'TE_{col}'] = X_encoded[f'TE_{col}'].astype('float32')
        
        return X_encoded
    
    def transform(self, X):
        X_encoded = X.copy()
        for col in self.cols:
            X_encoded[f'TE_{col}'] = X[col].map(self.encodings[col])
            X_encoded[f'TE_{col}'] = X_encoded[f'TE_{col}'].fillna(self.global_means[col]).astype('float32')
        return X_encoded

# Separate back into train, test, original
print("\nSeparating data into train/test/original...")

train_idx = combined['source'] == 'train'
test_idx = combined['source'] == 'test'
original_idx = combined['source'] == 'original'

X_train = combined[train_idx].reset_index(drop=True).copy()
X_test = combined[test_idx].reset_index(drop=True).copy()
X_original = combined[original_idx].reset_index(drop=True).copy()

# Get targets
y_train = X_train[TARGET].copy()
y_original = X_original[TARGET].copy()

# Memory optimization: delete combined to free up memory
del combined
gc.collect()

print(f"Memory after separation:")
print(f"X_train: {X_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"X_test: {X_test.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"X_original: {X_original.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Drop source and target columns
X_train = X_train.drop(columns=['source', TARGET])
X_test = X_test.drop(columns=['source', TARGET])
X_original = X_original.drop(columns=['source', TARGET])

print(f"\nFinal shapes:")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}, X_original shape: {X_original.shape}")


Separating data into train/test/original...
Memory after separation:
X_train: 302.62 MB
X_test: 129.45 MB
X_original: 10.21 MB

Final shapes:
X_train shape: (593994, 21), X_test shape: (254569, 21), X_original shape: (20000, 21)


In [5]:
# ============================================
# ADDITIONAL TARGET ENCODINGS (MEMORY OPTIMIZED)
# ============================================

def create_target_encodings(df_train, df_test, df_original, categorical_cols, target_col, y_original):
    """Create target encodings using original data"""
    print("\nCreating target encodings from original dataset...")
    encodings = {}
    
    # Use original data for TE
    df_original_temp = df_original.copy()
    df_original_temp[target_col] = y_original
    
    # Process in batches to save memory
    batch_size = 5
    for i in range(0, len(categorical_cols), batch_size):
        batch = categorical_cols[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(categorical_cols)-1)//batch_size + 1}")
        
        for col in batch:
            if col in df_original.columns:
                try:
                    # Mean encoding
                    te_map = df_original_temp.groupby(col)[target_col].mean()
                    df_train[f'TE_original_{col}_mean'] = df_train[col].map(te_map).astype('float32')
                    df_test[f'TE_original_{col}_mean'] = df_test[col].map(te_map).astype('float32')
                    
                    # Fill NaN with global mean
                    global_mean = y_original.mean()
                    df_train[f'TE_original_{col}_mean'] = df_train[f'TE_original_{col}_mean'].fillna(global_mean)
                    df_test[f'TE_original_{col}_mean'] = df_test[f'TE_original_{col}_mean'].fillna(global_mean)
                    
                    # Count encoding
                    te_count = df_original_temp.groupby(col).size()
                    df_train[f'TE_original_{col}_count'] = df_train[col].map(te_count).astype('int32')
                    df_test[f'TE_original_{col}_count'] = df_test[col].map(te_count).astype('int32')
                    df_train[f'TE_original_{col}_count'] = df_train[f'TE_original_{col}_count'].fillna(0)
                    df_test[f'TE_original_{col}_count'] = df_test[f'TE_original_{col}_count'].fillna(0)
                    
                    encodings[col] = te_map
                except Exception as e:
                    print(f"Error processing {col}: {e}")
        
        # Free memory
        gc.collect()
    
    del df_original_temp
    gc.collect()
    
    return df_train, df_test, encodings

def create_pseudo_target_encodings(df_train, df_test, df_original, y_original, pseudo_target_col, encode_cols):
    """Create target encodings using pseudo targets"""
    print(f"\nCreating pseudo-target encodings for {pseudo_target_col}...")
    
    # Combine train and original for encoding
    df_combined = pd.concat([df_original, df_train], axis=0).reset_index(drop=True)
    y_combined = pd.concat([pd.Series(y_original), pd.Series(np.zeros(len(df_train)))], axis=0)
    
    # Get pseudo target
    if pseudo_target_col in df_combined.columns:
        pseudo_target = df_combined[pseudo_target_col]
        
        # If numerical, bin it
        if pseudo_target.dtype in ['float32', 'float64', 'int32', 'int64']:
            pseudo_target = pd.qcut(pseudo_target, q=10, labels=False, duplicates='drop').fillna(0).astype('int8')
        
        # Process in batches
        batch_size = 5
        for i in range(0, min(20, len(encode_cols)), batch_size):  # Limit to 20 columns
            batch = encode_cols[i:i+batch_size]
            
            for col in batch:
                if col in df_combined.columns:
                    try:
                        # Calculate mean encoding
                        encoding = pd.Series(pseudo_target).groupby(df_combined[col]).mean()
                        
                        # Apply to train and test
                        df_train[f'TE_{pseudo_target_col}_{col}'] = df_train[col].map(encoding).astype('float32')
                        df_test[f'TE_{pseudo_target_col}_{col}'] = df_test[col].map(encoding).astype('float32')
                        
                        # Fill NaN
                        global_mean = pseudo_target.mean()
                        df_train[f'TE_{pseudo_target_col}_{col}'] = df_train[f'TE_{pseudo_target_col}_{col}'].fillna(global_mean)
                        df_test[f'TE_{pseudo_target_col}_{col}'] = df_test[f'TE_{pseudo_target_col}_{col}'].fillna(global_mean)
                    except Exception as e:
                        print(f"Error processing {col}: {e}")
            
            # Free memory
            gc.collect()
    
    # Clean up
    del df_combined, y_combined
    gc.collect()
    
    return df_train, df_test

# Apply target encodings
# Identify categorical columns (including engineered ones)
all_categorical = [col for col in X_train.columns 
                   if X_train[col].dtype == 'object' or 
                   X_train[col].nunique() < 100 or
                   'digit' in col or 'round' in col or 'qbin' in col]

# Limit to top categoricals to save memory
top_categorical = []
for col in all_categorical:
    if col in CATEGORICAL or 'grade_subgrade' in col or 'employment' in col:
        top_categorical.append(col)
    elif X_train[col].nunique() < 50:  # Low cardinality
        top_categorical.append(col)

top_categorical = top_categorical[:50]  # Limit to 50 columns

print(f"\nTarget encoding {len(top_categorical)} categorical columns...")

# 1. Target Encoding using original dataset
X_train, X_test, _ = create_target_encodings(
    X_train, X_test, X_original, 
    top_categorical[:30], TARGET, y_original
)

# 2. Pseudo-target encodings
X_train, X_test = create_pseudo_target_encodings(
    X_train, X_test, X_original, y_original,
    'employment_status', top_categorical[:20]
)

X_train, X_test = create_pseudo_target_encodings(
    X_train, X_test, X_original, y_original,
    'debt_to_income_ratio', top_categorical[:20]
)

print(f"\nFinal shapes after target encoding:")
print(f"X_train: {X_train.shape}, Memory: {X_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"X_test: {X_test.shape}, Memory: {X_test.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Clean up original data
del X_original
gc.collect()


Target encoding 16 categorical columns...

Creating target encodings from original dataset...
Processing batch 1/4
Processing batch 2/4
Error processing age: Cannot convert non-finite values (NA or inf) to integer
Error processing monthly_income: Cannot convert non-finite values (NA or inf) to integer
Error processing loan_term: Cannot convert non-finite values (NA or inf) to integer
Error processing installment: Cannot convert non-finite values (NA or inf) to integer
Processing batch 3/4
Error processing num_of_open_accounts: Cannot convert non-finite values (NA or inf) to integer
Error processing total_credit_limit: Cannot convert non-finite values (NA or inf) to integer
Error processing current_balance: Cannot convert non-finite values (NA or inf) to integer
Error processing delinquency_history: Cannot convert non-finite values (NA or inf) to integer
Error processing public_records: Cannot convert non-finite values (NA or inf) to integer
Processing batch 4/4
Error processing num_of

0

In [6]:
# ============================================
# PREPARE FOR MODELING (MEMORY OPTIMIZED)
# ============================================

print("\nPreparing data for modeling...")

# Convert remaining object columns to category codes
object_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']
print(f"Converting {len(object_cols)} object columns...")

for col in object_cols:
    # Factorize to save memory
    X_train[col], unique = pd.factorize(X_train[col])
    X_test[col] = X_test[col].map(dict(zip(unique, range(len(unique)))))
    X_test[col] = X_test[col].fillna(-1).astype('int32')
    
    # Convert to appropriate dtype based on cardinality
    if X_train[col].max() < 128:
        X_train[col] = X_train[col].astype('int8')
        X_test[col] = X_test[col].astype('int8')
    elif X_train[col].max() < 32768:
        X_train[col] = X_train[col].astype('int16')
        X_test[col] = X_test[col].astype('int16')
    else:
        X_train[col] = X_train[col].astype('int32')
        X_test[col] = X_test[col].astype('int32')

# Fill any remaining NaN
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# Final memory optimization: downcast float64 to float32
for col in X_train.select_dtypes(include=['float64']).columns:
    X_train[col] = X_train[col].astype('float32')
    X_test[col] = X_test[col].astype('float32')

print(f"\nFinal data preparation complete:")
print(f"X_train shape: {X_train.shape}, Memory: {X_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"X_test shape: {X_test.shape}, Memory: {X_test.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Free memory
gc.collect()


Preparing data for modeling...
Converting 6 object columns...

Final data preparation complete:
X_train shape: (593994, 59), Memory: 122.36 MB
X_test shape: (254569, 59), Memory: 52.44 MB


0

In [7]:
# ============================================
# XGBOOST MODEL TRAINING (MEMORY OPTIMIZED)
# ============================================

# XGBoost parameters optimized for memory
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 6,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.5,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'n_estimators': 10000,
    'early_stopping_rounds': 200,
    'random_state': SEED,
    'n_jobs': -1,
    'enable_categorical': False,  # We've already encoded categories
    'tree_method': 'hist',  # More memory efficient
    'max_bin': 256,  # Reduce for memory
    'grow_policy': 'lossguide',
}

# Cross-validation setup
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage for predictions
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
feature_importances = []

print(f"\nStarting {N_FOLDS}-fold cross-validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f"\n--- Fold {fold}/{N_FOLDS} ---")
    
    # Split data
    X_tr = X_train.iloc[train_idx].reset_index(drop=True).copy()
    X_val = X_train.iloc[val_idx].reset_index(drop=True).copy()
    y_tr = y_train.iloc[train_idx].reset_index(drop=True).copy()
    y_val = y_train.iloc[val_idx].reset_index(drop=True).copy()
    
    print(f"Train: {X_tr.shape}, Validation: {X_val.shape}")
    
    # Train XGBoost
    model = XGBClassifier(**xgb_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=1000
    )
    
    # Predict
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    
    # Predict on test
    test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # Store feature importance
    fold_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': model.feature_importances_,
        'fold': fold
    })
    feature_importances.append(fold_importance)
    
    # Score
    fold_score = roc_auc_score(y_val, val_preds)
    print(f"Fold {fold} AUC: {fold_score:.6f}")
    
    # Clean up to save memory
    del model, X_tr, X_val, y_tr, y_val
    gc.collect()

# Calculate overall score
overall_auc = roc_auc_score(y_train, oof_preds)
print(f"\n{'='*60}")
print(f"Overall OOF AUC: {overall_auc:.6f}")
print(f"{'='*60}")

# Combine feature importances
feature_importance_df = pd.concat(feature_importances, ignore_index=True)
mean_importance = feature_importance_df.groupby('feature')['importance'].mean().sort_values(ascending=False)

print("\nTop 20 most important features:")
for i, (feat, imp) in enumerate(mean_importance.head(20).items(), 1):
    print(f"{i:2d}. {feat:40s} | Importance: {imp:.4f}")

# Free memory
del feature_importance_df, mean_importance
gc.collect()


Starting 5-fold cross-validation...

--- Fold 1/5 ---
Train: (475195, 59), Validation: (118799, 59)
[0]	validation_0-auc:0.85757
[1000]	validation_0-auc:0.92065
[2000]	validation_0-auc:0.92209
[3000]	validation_0-auc:0.92266
[4000]	validation_0-auc:0.92284
[4750]	validation_0-auc:0.92287
Fold 1 AUC: 0.922880

--- Fold 2/5 ---
Train: (475195, 59), Validation: (118799, 59)
[0]	validation_0-auc:0.85691
[1000]	validation_0-auc:0.92015
[2000]	validation_0-auc:0.92162
[3000]	validation_0-auc:0.92227
[4000]	validation_0-auc:0.92258
[5000]	validation_0-auc:0.92268
[5705]	validation_0-auc:0.92273
Fold 2 AUC: 0.922732

--- Fold 3/5 ---
Train: (475195, 59), Validation: (118799, 59)
[0]	validation_0-auc:0.85614
[1000]	validation_0-auc:0.91850
[2000]	validation_0-auc:0.91989
[3000]	validation_0-auc:0.92050
[4000]	validation_0-auc:0.92082
[5000]	validation_0-auc:0.92091
[5454]	validation_0-auc:0.92092
Fold 3 AUC: 0.920921

--- Fold 4/5 ---
Train: (475195, 59), Validation: (118799, 59)
[0]	validatio

0